In [1]:
import seaborn as sns
import pandas as pd
import re
import numpy as np
from urllib.parse import urlparse

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)

%matplotlib inline


Legitimate/true = 1
Phishing/false = 0

In [2]:
DATA_PATH = './malicious_phish.csv'
df1 = pd.read_csv(DATA_PATH)
print(df1.head(5))

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [3]:
df1 = df1[df1['type'] != 'defacement']
df1 = df1[df1['type'] != 'malware']
print(df1.head(5))
print(df1['type'].value_counts())

                                                 url      type
0                                   br-icloud.com.br  phishing
1                mp3raid.com/music/krizz_kaliko.html    benign
2                    bopsecrets.org/rexroth/cr/1.htm    benign
5  http://buzzfil.net/m/show-art/ils-etaient-loin...    benign
6      espn.go.com/nba/player/_/id/3457/brandon-rush    benign
type
benign      428103
phishing     94111
Name: count, dtype: int64


In [4]:
df1['type'] = (df1['type'] == 'benign'). astype(int)
df1['label'] = df1['type'].astype(int)
df1 = df1.drop(columns=["type"])
print(df1.head(5))
print(df1['label'].value_counts())

                                                 url  label
0                                   br-icloud.com.br      0
1                mp3raid.com/music/krizz_kaliko.html      1
2                    bopsecrets.org/rexroth/cr/1.htm      1
5  http://buzzfil.net/m/show-art/ils-etaient-loin...      1
6      espn.go.com/nba/player/_/id/3457/brandon-rush      1
label
1    428103
0     94111
Name: count, dtype: int64


In [28]:
DATA_PATH = './new_data_urls.csv'
data_set_1 = pd.read_csv(DATA_PATH)
print(data_set_1.head(5))
DATA_PATH = './url_dataset.csv'
data_set_2 = pd.read_csv(DATA_PATH)
print(data_set_2.head(5))
print("\nDtypes:"); print(data_set_1.dtypes)


                                       url  status
0  0000111servicehelpdesk.godaddysites.com     0.0
1     000011accesswebform.godaddysites.com     0.0
2                             00003.online     0.0
3      0009servicedeskowa.godaddysites.com     0.0
4                     000n38p.wcomhost.com     0.0
                         url        type
0     https://www.google.com  legitimate
1    https://www.youtube.com  legitimate
2   https://www.facebook.com  legitimate
3      https://www.baidu.com  legitimate
4  https://www.wikipedia.org  legitimate

Dtypes:
url           str
status    float64
dtype: object


In [29]:
data_set_2['type'] = (data_set_2['type'] == 'legitimate'). astype(int)
data_set_2['label'] = data_set_2['type'].astype(int)
data_set_2 = data_set_2.drop(columns=["type"])
data_set_1 = data_set_1.dropna()
data_set_1['label'] = data_set_1['status'].astype(int)
data_set_1 = data_set_1.drop(columns=["status"])
print(data_set_1.head(5))
print(data_set_2.head(5))

                                       url  label
0  0000111servicehelpdesk.godaddysites.com      0
1     000011accesswebform.godaddysites.com      0
2                             00003.online      0
3      0009servicedeskowa.godaddysites.com      0
4                     000n38p.wcomhost.com      0
                         url  label
0     https://www.google.com      1
1    https://www.youtube.com      1
2   https://www.facebook.com      1
3      https://www.baidu.com      1
4  https://www.wikipedia.org      1


set data_set_2 type column to int, to match the style of the others.         
rename data_type_1 column status to type.         
delete the status column.         

In [30]:
df = pd.concat([data_set_1, data_set_2, df1], axis=0, ignore_index=True)
print(df)
print(df['label'].value_counts())

                                                   url  label
0              0000111servicehelpdesk.godaddysites.com      0
1                 000011accesswebform.godaddysites.com      0
2                                         00003.online      0
3                  0009servicedeskowa.godaddysites.com      0
4                                 000n38p.wcomhost.com      0
...                                                ...    ...
1381762        xbox360.ign.com/objects/850/850402.html      0
1381763   games.teamxbox.com/xbox-360/1860/Dead-Space/      0
1381764     www.gamespot.com/xbox360/action/deadspace/      0
1381765  en.wikipedia.org/wiki/Dead_Space_(video_game)      0
1381766      www.angelfire.com/goth/devilmaycrytonite/      0

[1381767 rows x 2 columns]
label
1    914035
0    467732
Name: count, dtype: int64


combine both datasets and print out both the datset and a count of the two types

In [31]:
df.drop_duplicates(inplace=True)
df['url'] = df['url'].str.lower()
print(df['label'].value_counts())


label
1    829553
0    449189
Name: count, dtype: int64


Delete Duplicates and set all letters to lowercase                    
Setting all letters to lowercase ensure easire comparison later on, and while URL's are case sensitive past the domain, for our purpose, it should be fine

In [34]:
def safe_extract_domain(url):
    if pd.isna(url):
        return None
    try:
        # Ensure it has a scheme so urlparse reads netloc correctly
        if not re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', url):
            url = 'http://' + url
        parsed = urlparse(url)
        # Strip port and user info if needed, or return netloc
        netloc = parsed.netloc.split('@')[-1].split(':')[0]
        return netloc if netloc else None
    except ValueError:
        # Fallback to regex if urllib throws Invalid IPv6 URL error
        match = re.search(r'://([^/]+)', url)
        if match:
            return match.group(1).split('@')[-1].split(':')[0]
        return None

df['domain'] = df['url'].apply(safe_extract_domain)
df = df.fillna('0')
print("\nDtypes:"); print(df.dtypes)
print(df.head(5))


Dtypes:
url             str
label         int64
domain          str
url_length    int64
dtype: object
                                       url  label                                   domain  url_length
0  0000111servicehelpdesk.godaddysites.com      0  0000111servicehelpdesk.godaddysites.com          39
1     000011accesswebform.godaddysites.com      0     000011accesswebform.godaddysites.com          36
2                             00003.online      0                             00003.online          12
3      0009servicedeskowa.godaddysites.com      0      0009servicedeskowa.godaddysites.com          35
4                     000n38p.wcomhost.com      0                     000n38p.wcomhost.com          20


In [35]:
df['url_length'] = df['url'].str.len()
df['domain_length'] = np.where(df['domain'] == '0', 0, df['domain'].apply(len))
df['is_https'] = df['url'].str.match(r'^https://').astype(int)

strict_ip_regex = r"^((25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\.){3}(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])$"
df['is_ip'] = df['url'].str.match(strict_ip_regex).astype(int)


print("\nDtypes:"); print(df.dtypes)
print(df.head(5))


Dtypes:
url                str
label            int64
domain             str
url_length       int64
domain_length    int64
is_https         int64
is_ip            int64
dtype: object
                                       url  label                                   domain  url_length  domain_length  is_https  is_ip
0  0000111servicehelpdesk.godaddysites.com      0  0000111servicehelpdesk.godaddysites.com          39             39         0      0
1     000011accesswebform.godaddysites.com      0     000011accesswebform.godaddysites.com          36             36         0      0
2                             00003.online      0                             00003.online          12             12         0      0
3      0009servicedeskowa.godaddysites.com      0      0009servicedeskowa.godaddysites.com          35             35         0      0
4                     000n38p.wcomhost.com      0                     000n38p.wcomhost.com          20             20         0      0


In [36]:

df['letter_count'] = df['url'].str.findall(r'[a-zA-Z]').str.len()
df['digit_count'] = df['url'].str.findall(r'[0-9]').str.len()
df['special_count'] = df['url'].str.findall(r'[^a-zA-Z0-9\s]').str.len()

df['eq_count'] = df['url'].str.findall(r'[=]').str.len()
df['qm_count'] = df['url'].str.findall(r'[?]').str.len()
df['amp_count'] = df['url'].str.findall(r'[&]').str.len()
df['dot_count'] = df['url'].str.findall(r'[.]').str.len()
df['dash_count'] = df['url'].str.findall(r'[-]').str.len()
df['under_count'] = df['url'].str.findall(r'[_]').str.len()


print("\nDtypes:"); print(df.dtypes)
print(df['is_https'].value_counts())
print(df['is_ip'].value_counts())
print(df.head(5))



Dtypes:
url                str
label            int64
domain             str
url_length       int64
domain_length    int64
is_https         int64
is_ip            int64
letter_count     int64
digit_count      int64
special_count    int64
eq_count         int64
qm_count         int64
amp_count        int64
dot_count        int64
dash_count       int64
under_count      int64
dtype: object
is_https
0    806527
1    472215
Name: count, dtype: int64
is_ip
0    1277773
1        969
Name: count, dtype: int64
                                       url  label                                   domain  url_length  domain_length  is_https  is_ip  letter_count  \
0  0000111servicehelpdesk.godaddysites.com      0  0000111servicehelpdesk.godaddysites.com          39             39         0      0            30   
1     000011accesswebform.godaddysites.com      0     000011accesswebform.godaddysites.com          36             36         0      0            28   
2                             00003.

In [37]:
df.to_csv('output.csv', index=False)